# 03 - Construção da tabela fato

---
## Objetivo

Este notebook constrói a tabela fato de Assistência Estudantil (schema 2023),
a partir da extração agregada do SEDAP+ e das dimensões já processadas em
`02_construcao_dimensoes.ipynb`.

---

## Importação das bibliotecas necessárias

In [1]:
import sys
from pathlib import Path
import pandas as pd

In [2]:
# Define a pasta raiz do projeto (um nível acima da pasta /notebooks)
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Adiciona a raiz ao sys.path para o Python encontrar a pasta 'src'
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

# Caminhos dos arquivos de entrada
# Ajuste o nome do arquivo se a extração do SEDAP+ tiver outro nome na sua pasta raw.
PATH_RAW_FATO = BASE_DIR / "data" / "raw" / "SEDAP" / "fato_assistencia_raw.csv"

PATH_DIM = BASE_DIR / "data" / "processed" / "Dimensões"
PATH_DIM_CURSO_CAMPUS1 = PATH_DIM / "dim_curso_ufpb_campus_1.csv"
PATH_DIM_CURSO_CENTRO = PATH_DIM / "dim_curso_centro.csv"
PATH_DIM_CENTRO = PATH_DIM / "dim_centro.csv"
PATH_DIM_SEXO = PATH_DIM / "dim_sexo.csv"
PATH_DIM_RACA = PATH_DIM / "dim_raca.csv"
PATH_DIM_TURNO = PATH_DIM / "dim_turno.csv"
PATH_DIM_GRAU = PATH_DIM / "dim_grau.csv"
PATH_DIM_MODALIDADE = PATH_DIM / "dim_modalidade.csv"

# Caminho de saída
PATH_FATO = BASE_DIR / "data" / "processed" / "Fato" / "fato_assistencia.csv"

---
## 1. Carregar as dimensões processadas

Todas já validadas em `02_construcao_dimensoes.ipynb` (schema 2023).

In [3]:
dim_curso_campus1 = pd.read_csv(PATH_DIM_CURSO_CAMPUS1, sep=";", encoding="utf-8-sig")
dim_curso_centro = pd.read_csv(PATH_DIM_CURSO_CENTRO, sep=";", encoding="utf-8-sig")
dim_centro = pd.read_csv(PATH_DIM_CENTRO, sep=";", encoding="utf-8-sig")
dim_sexo = pd.read_csv(PATH_DIM_SEXO, sep=";", encoding="utf-8-sig")
dim_raca = pd.read_csv(PATH_DIM_RACA, sep=";", encoding="utf-8-sig")
dim_turno = pd.read_csv(PATH_DIM_TURNO, sep=";", encoding="utf-8-sig")
dim_grau = pd.read_csv(PATH_DIM_GRAU, sep=";", encoding="utf-8-sig")
dim_modalidade = pd.read_csv(PATH_DIM_MODALIDADE, sep=";", encoding="utf-8-sig")

print(f"dim_curso_campus1: {len(dim_curso_campus1)} cursos")
print(f"dim_curso_centro:  {len(dim_curso_centro)} cursos")
print(f"dim_centro:        {len(dim_centro)} centros")

dim_curso_campus1: 94 cursos
dim_curso_centro:  94 cursos
dim_centro:        13 centros


---
## 2. Carregar a fato bruta (SEDAP+)

Extração agregada (schema 2023) — ver `01_extracao_sedap.ipynb`, consulta
"4.2 Construção Fato".

In [4]:
fato_raw = pd.read_csv(PATH_RAW_FATO, encoding="utf-8-sig")

print(f"Linhas: {len(fato_raw)}")
print(f"Cursos únicos (todos os campi): {fato_raw['CO_CURSO'].nunique()}")
display(fato_raw.head())

Linhas: 3232
Cursos únicos (todos os campi): 120


,CO_CURSO,TP_SEXO,TP_COR_RACA,TP_TURNO,IN_RESERVA_VAGAS,IN_APOIO_SOCIAL,IN_APOIO_ALIMENTACAO,IN_APOIO_MORADIA,IN_APOIO_TRANSPORTE,IN_APOIO_MATERIAL_DIDATICO,IN_APOIO_BOLSA_PERMANENCIA,IN_APOIO_BOLSA_TRABALHO,TOTAL_ALUNOS
0,13394,1,3,4.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,12
1,13394,1,3,4.0,0,1,1.0,0.0,0.0,0.0,0.0,0.0,2
2,13394,1,3,4.0,1,1,1.0,1.0,0.0,0.0,0.0,0.0,2
3,13394,1,1,4.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,18
4,13394,1,2,3.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2


---
## 3. Construir a Fato

Usa `criar_fato_assistencia` (`src/fato/fato_assistencia.py`):
- filtro do Campus I via whitelist contra `dim_curso_campus1` (sem lista manual);
- turno nulo -> `ID_TURNO = 0` ("Não informado");
- IDPNA a partir de `IN_RESERVA_VAGAS` (disponível na extração 2023);
- merge único de `ID_CENTRO`, `ID_GRAU` e `ID_MODALIDADE`, com validação
  automática de cursos órfãos.

In [6]:
import importlib
import src.facts.fato_assistencia as fa

importlib.reload(fa)
from src.facts.fato_assistencia import criar_fato_assistencia

fato = criar_fato_assistencia(
    fato_raw,
    dim_curso_campus1,
    dim_curso_centro,
    dim_grau,
    dim_modalidade,
)

display(fato.head(10))
fato.info()

print("\nValores Nulos:\n", fato.isna().sum())
print("\nCursos únicos:", fato["CO_CURSO"].nunique())
print("Centros únicos:", fato["ID_CENTRO"].nunique())

# Asserts de segurança
assert fato["CO_CURSO"].nunique() == len(dim_curso_campus1), \
    "Nem todos os cursos do Campus I aparecem na fato"
assert fato.isna().sum().sum() == 0, "Existem valores nulos na fato final"

,CO_CURSO,ID_CENTRO,ID_GRAU,ID_MODALIDADE,ID_SEXO,ID_RACA,ID_TURNO,IN_RESERVA_VAGAS,IN_APOIO_SOCIAL,IN_APOIO_ALIMENTACAO,IN_APOIO_MORADIA,IN_APOIO_TRANSPORTE,IN_APOIO_MATERIAL_DIDATICO,IN_APOIO_BOLSA_PERMANENCIA,IN_APOIO_BOLSA_TRABALHO,TOTAL_ALUNOS,RECEBE_AUXILIO,IDPNA,TOTAL_IDPNA
0,13394,5,1,1,1,3,4,0,0,0,0,0,0,0,0,12,0,0,0
1,13394,5,1,1,1,3,4,0,1,1,0,0,0,0,0,2,1,0,0
2,13394,5,1,1,1,3,4,1,1,1,1,0,0,0,0,2,1,0,0
3,13394,5,1,1,1,1,4,0,0,0,0,0,0,0,0,18,0,0,0
4,13394,5,1,1,1,2,3,0,0,0,0,0,0,0,0,2,0,0,0
5,13394,5,1,1,2,3,3,1,1,1,0,0,0,0,0,4,1,0,0
6,13394,5,1,1,1,2,4,1,1,1,0,0,0,0,0,2,1,0,0
7,13394,5,1,1,2,1,4,0,1,0,0,0,0,1,0,2,1,0,0
8,13394,5,1,1,2,3,4,1,1,1,0,0,0,0,0,4,1,0,0
9,13394,5,1,1,1,2,4,0,0,0,0,0,0,0,0,3,0,0,0


<class 'pandas.DataFrame'>
RangeIndex: 2468 entries, 0 to 2467
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   CO_CURSO                    2468 non-null   int64
 1   ID_CENTRO                   2468 non-null   int64
 2   ID_GRAU                     2468 non-null   int64
 3   ID_MODALIDADE               2468 non-null   int64
 4   ID_SEXO                     2468 non-null   int64
 5   ID_RACA                     2468 non-null   int64
 6   ID_TURNO                    2468 non-null   int64
 7   IN_RESERVA_VAGAS            2468 non-null   int64
 8   IN_APOIO_SOCIAL             2468 non-null   int64
 9   IN_APOIO_ALIMENTACAO        2468 non-null   int64
 10  IN_APOIO_MORADIA            2468 non-null   int64
 11  IN_APOIO_TRANSPORTE         2468 non-null   int64
 12  IN_APOIO_MATERIAL_DIDATICO  2468 non-null   int64
 13  IN_APOIO_BOLSA_PERMANENCIA  2468 non-null   int64
 14  IN_APOIO_BOLSA_TRAB

---
## 4. Conferência cruzada com as demais dimensões

Confere se toda chave estrangeira da Fato existe na respectiva dimensão
(nenhuma linha "órfã").

In [7]:
checagens = {
    "ID_SEXO -> dim_sexo": (fato["ID_SEXO"], dim_sexo["ID_SEXO"]),
    "ID_RACA -> dim_raca": (fato["ID_RACA"], dim_raca["ID_RACA"]),
    "ID_TURNO -> dim_turno": (fato["ID_TURNO"], dim_turno["ID_TURNO"]),
    "ID_CENTRO -> dim_centro": (fato["ID_CENTRO"], dim_centro["ID_CENTRO"]),
    "ID_GRAU -> dim_grau": (fato["ID_GRAU"], dim_grau["ID_GRAU"]),
    "ID_MODALIDADE -> dim_modalidade": (fato["ID_MODALIDADE"], dim_modalidade["ID_MODALIDADE"]),
}

for nome, (chave_fato, chave_dim) in checagens.items():
    orfaos = set(chave_fato) - set(chave_dim)
    status = "OK" if not orfaos else f"FALTANDO: {sorted(orfaos)}"
    print(f"{nome}: {status}")
    assert not orfaos, f"Chave órfã encontrada em {nome}"

ID_SEXO -> dim_sexo: OK
ID_RACA -> dim_raca: OK
ID_TURNO -> dim_turno: OK
ID_CENTRO -> dim_centro: OK
ID_GRAU -> dim_grau: OK
ID_MODALIDADE -> dim_modalidade: OK


---
## 5. Indicadores de conferência rápida

In [8]:
print("Total de alunos:", fato["TOTAL_ALUNOS"].sum())
print()
print("IDPNA (alunos):")
print(fato.groupby("IDPNA")["TOTAL_ALUNOS"].sum())
print()
print("RECEBE_AUXILIO (alunos):")
print(fato.groupby("RECEBE_AUXILIO")["TOTAL_ALUNOS"].sum())

Total de alunos: 31210

IDPNA (alunos):
IDPNA
0    19480
1    11730
Name: TOTAL_ALUNOS, dtype: int64

RECEBE_AUXILIO (alunos):
RECEBE_AUXILIO
0    28573
1     2637
Name: TOTAL_ALUNOS, dtype: int64


---
## 6. Exportação

In [ ]:
PATH_FATO.parent.mkdir(parents=True, exist_ok=True)

fato.to_csv(
    PATH_FATO,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print(f"✅ Fato Assistência Estudantil salva com sucesso em: {PATH_FATO}")